# CS604 Final Training: ViT-B/16 Transfer to 10,000-Class Fine-Grained Classification (Colab)

This is the **final training** version, set up from the short-run results. The training code is exactly the same as the experiment notebook (`cs604_vit_adaptation.ipynb`); only the final config is preset. One "Run all" does two training runs in order, and each one creates a submission file you can upload to CodaBench.

**Short-run results** (batch `0928_0636`, A100-80GB; the local holdout set is 2 images per class taken from the training set, 20,000 images in total):

| Method | Local holdout top-1 | top-5 | Trainable params (excl. head) | GPU time | Peak VRAM | Doc reference |
|---|---|---|---|---|---|---|
| Zero-shot (class prototypes) | 43.95% | 68.05% | 0 | 0.11 h | 3.9 GB | 43.7% |
| LoRA, 3 epochs | 68.57% | 86.96% | 294,912 (0.344%) | 0.40 h | 13.5 GB | 68.3% |
| Full fine-tuning, 3 epochs | 70.88% | 88.45% | 85,798,656 (100%) | 0.45 h | 18.9 GB | 70.8% |

**Final config based on these results**
1. **Full fine-tuning, 8 epochs** (main CodaBench submission): highest accuracy in the short runs. Holdout accuracy over the three epochs was 68.3% → 70.6% → 70.9%, only +0.3% in the last epoch. Also, the training loss drops by a step at the start of each epoch and barely moves within an epoch, which means the model is memorizing training images it has already seen (only 43 per class). So we only make it a bit longer (8 epochs) and do not use the full time budget, to avoid overfitting.
2. **LoRA, 10 epochs, head LR lowered from 1e-3 to 2e-4** (the parameter-efficient method for the report): holdout accuracy 62.4% → 67.0% → 68.6%, still clearly rising at epoch 3, so we train longer. Also, while the LR warmed up to 1e-3, the training loss went up from 2.2 to 2.6, which means the LR was too high. In full fine-tuning the head trained smoothly with 1e-4, so we lower the LoRA head LR to 2e-4 and keep 1e-3 for the LoRA modules. This change has not been tested on its own yet; trust the local holdout and CodaBench scores from this run.
3. **Keep a local holdout set** (2 images per class): to check that longer training does not overfit, and to get accuracy curves for the report.

**How long it takes**: on A100-80GB, about 1 hour for full fine-tuning and about 1 hour for LoRA, both within the 1.2-hour budget per run (96 GB·h ÷ 80 GB). If it runs slower, the LR schedule is shrunk automatically after training starts. The whole batch takes about 2–2.5 hours (plus 5–15 min to extract the data from Drive). Keep the page open the whole time.

**How to use**
1. The data is already saved in `cs604/data/` on Drive by `cs604_download_to_drive.ipynb`.
2. `Runtime → Change runtime type`: choose **A100 GPU**, then `Runtime → Run all`, and allow Google Drive access when asked.
3. Results are saved on Drive: each run's submission files and logs are in `cs604/runs/<run name>/`, and the comparison table and plot are in `cs604/batches/<BATCH_NAME>/`.
4. If you get disconnected: set `BATCH_NAME` to the name printed at the start of the log and "Run all" again. Finished runs are skipped, and an interrupted one resumes from its latest epoch.

**Notes**
- **Training time budget** is converted as VRAM × hours = 96 GB·h and counted per experiment: 1.2 hours on A100-80GB, 2.4 hours on A100-40GB. If the TA confirms that 2 hours is fine, set `TIME_BUDGET_H` to 2 and increase `EPOCHS` a bit.
- **Parameter counts** follow the project doc and do not include the classifier head. The 10,000-class head has another 7.69M params, listed separately.
- **Base model**: by default we use the IN21K→IN1K version given on page 1 of the doc. Please confirm with the TA.

## 1. Config

In [ ]:
TEAM_ID = "06"          # team number; the submission file is val_team_{TEAM_ID}.txt

# ===== Final training (based on the short runs in batch 0928_0636; see the notes at the top) =====
EXPERIMENTS = [
    # Full fine-tuning: best holdout score in the short runs (70.88%), almost flat by the last epoch → a bit longer, 8 epochs (main CodaBench submission)
    dict(METHOD="full", EPOCHS=8),
    # LoRA: 68.57% in the short run, still rising at epoch 3 → longer, 10 epochs;
    # loss went up while the LR warmed up to 1e-3 → head LR lowered to 2e-4 (LoRA modules keep 1e-3)
    dict(METHOD="lora", EPOCHS=10, HEAD_LR=2e-4),
]
RUN_FINAL = False       # True = after the runs above, pick the method with the best holdout score, apply FINAL_OVERRIDES and train once more (good for unattended runs; better to check the results first and then choose settings)
FINAL_METHOD = None     # None = pick automatically (excluding linear); or set one, e.g. "lora"
FINAL_OVERRIDES = dict(EPOCHS=15, WARMUP_EPOCHS=1, HOLDOUT_PER_CLASS=0)  # final training: more epochs (shortened automatically if over budget), use all 45 images per class
BATCH_NAME = None       # None = auto name; to resume after a disconnect, set the previous BATCH_NAME
LOAD_RUN = None         # name of a trained run: skip training, just load the weights and predict (for Phase 2)
PREDICT_SPLITS = ["val"]  # change to ["test"] (or ["val", "test"]) after the Phase 2 test set is released

# ===== Defaults (each experiment can override them in EXPERIMENTS) =====
METHOD = "lora"         # 'lora' | 'full' | 'adaptformer' | 'adapter' | 'bitfit' | 'partial' | 'linear'
MODEL_NAME = "vit_base_patch16_224.augreg2_in21k_ft_in1k"  # timm version given on page 1 of the doc (IN21K→IN1K)
# MODEL_NAME = "vit_base_patch16_224.orig_in21k"            # = HF google/vit-base-patch16-224-in21k (pretrained on IN21K only)
HOLDOUT_PER_CLASS = 2   # images per class held out from train for local validation (the official val set has no labels)
EPOCHS = 10             # max number of epochs; shortened automatically if over the time budget
TIME_BUDGET_H = None    # None = convert from 96 GB·h automatically; or set the hours directly
HEAD_INIT = "lp"        # 'lp' linear probe | 'proto' class prototypes (zero-shot) | 'random'
TRAIN_HEAD = True       # whether PEFT methods also train the classifier head

# ===== Hyperparameters (None = use the value in METHOD_DEFAULTS below) =====
BATCH_SIZE = None       # per-step batch; None = pick based on VRAM
EFFECTIVE_BATCH = 256   # effective batch: if one step can't fit it, use gradient accumulation to reach it
LR = None
WEIGHT_DECAY = None
HEAD_LR = None          # None = same as LR
LABEL_SMOOTHING = 0.1
WARMUP_EPOCHS = 1
RRC_SCALE = (0.25, 1.0)  # area range for RandomResizedCrop
LORA_R = 8
LORA_ALPHA = 16
LORA_TARGETS = ["qkv"]   # options: 'qkv', 'proj', 'fc1', 'fc2'
ADAPTER_DIM = 64         # bottleneck size for AdaptFormer / Adapter
PARTIAL_LAST_K = 2       # 'partial': train only the last K blocks
LP_EPOCHS = 10           # linear probe epochs (trained on cached features, so it is fast)
TTA_FLIP = False         # add a horizontal flip at prediction time (doubles inference latency)
COMPILE = False          # torch.compile; usually 10–20% faster on A100, the first compile takes 1–2 min
CKPT_EVERY = 1           # save a checkpoint every N epochs
LOG_EVERY = 100          # print a log line every N steps
SEED = 0

METHOD_DEFAULTS = {
    #                   LR        weight decay  layer-wise LR decay  DropPath
    "full":        dict(lr=1e-4, wd=0.05, layer_decay=0.75, drop_path=0.1),
    "partial":     dict(lr=3e-4, wd=0.05, layer_decay=1.0, drop_path=0.0),
    "lora":        dict(lr=1e-3, wd=1e-4, layer_decay=1.0, drop_path=0.0),
    "adaptformer": dict(lr=1e-3, wd=1e-4, layer_decay=1.0, drop_path=0.0, adapter_scale=0.1),
    "adapter":     dict(lr=1e-3, wd=1e-4, layer_decay=1.0, drop_path=0.0, adapter_scale=1.0),
    "bitfit":      dict(lr=1e-3, wd=0.0, layer_decay=1.0, drop_path=0.0),
    "linear":      dict(lr=0.0, wd=0.0, layer_decay=1.0, drop_path=0.0),
}
NUM_CLASSES = 10000

# ===== Paths and cleanup =====
DRIVE_DIR = "/content/drive/MyDrive/cs604"   # results and checkpoints are saved here
DRIVE_DATA_DIR = f"{DRIVE_DIR}/data"         # where the dataset is on Drive (save it there first with cs604_download_to_drive.ipynb)
DATA_DIR = "/content/data"                   # each session extracts the data to the local Colab disk; training reads images from here
DISCONNECT_WHEN_DONE = False  # disconnect the runtime when done to stop using compute units (True is recommended when running in the background)

## 2. Setup

In [ ]:
# Colab already has torch / torchvision; install timm and peft here
!pip install -q timm peft
# Colab's preinstalled torchao is too old, and new peft raises ImportError when creating LoRA. We don't need torchao, so just uninstall it
!pip uninstall -y -q torchao

In [ ]:
# Config variables come in two kinds: ones each experiment can override (RUN_KEYS), and ones shared by the whole batch (QUEUE_KEYS)
RUN_KEYS = ("METHOD", "MODEL_NAME", "HOLDOUT_PER_CLASS", "EPOCHS", "TIME_BUDGET_H", "HEAD_INIT", "TRAIN_HEAD",
            "BATCH_SIZE", "EFFECTIVE_BATCH", "LR", "WEIGHT_DECAY", "HEAD_LR", "LABEL_SMOOTHING", "WARMUP_EPOCHS",
            "RRC_SCALE", "LORA_R", "LORA_ALPHA", "LORA_TARGETS", "ADAPTER_DIM", "PARTIAL_LAST_K", "LP_EPOCHS",
            "TTA_FLIP", "COMPILE", "CKPT_EVERY", "LOG_EVERY", "SEED", "METHOD_DEFAULTS", "NUM_CLASSES")
QUEUE_KEYS = ("TEAM_ID", "EXPERIMENTS", "RUN_FINAL", "FINAL_METHOD", "FINAL_OVERRIDES", "BATCH_NAME", "LOAD_RUN",
              "PREDICT_SPLITS", "DRIVE_DIR", "DRIVE_DATA_DIR", "DATA_DIR", "DISCONNECT_WHEN_DONE")
BASE_CONFIG = {k: globals()[k] for k in RUN_KEYS}
QUEUE_CONFIG = {k: globals()[k] for k in QUEUE_KEYS}

import os, re, gc, sys, json, math, time, types, random, shutil, zipfile, subprocess, contextlib
from datetime import datetime
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

IN_COLAB = "google.colab" in sys.modules

# ---- Mount Google Drive: data is read from here, and results and checkpoints are saved here, so nothing is lost on disconnect ----
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = DRIVE_DIR
else:
    OUT_ROOT = os.path.abspath("outputs")

# ---- Device and mixed precision ----
if torch.cuda.is_available():
    DEVICE = "cuda"
    props = torch.cuda.get_device_properties(0)
    GPU_NAME = props.name
    gib = props.total_memory / 1024**3
    VRAM_GB = next((v for v in (8, 12, 16, 24, 32, 40, 48, 80, 94, 96, 141, 192) if v >= 0.98 * gib), round(gib))  # nominal VRAM
    AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16  # T4/V100 have no bf16
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
    GPU_NAME, VRAM_GB, AMP_DTYPE = DEVICE, None, None
AMP_NAME = str(AMP_DTYPE).replace("torch.", "") if AMP_DTYPE else "fp32"
NUM_WORKERS = min(16, os.cpu_count() or 2)

BATCH_NAME = BATCH_NAME or datetime.now().strftime("%m%d_%H%M")
BATCH_DIR = os.path.join(OUT_ROOT, "batches", BATCH_NAME)
os.makedirs(BATCH_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
with open(os.path.join(BATCH_DIR, "queue_config.json"), "w") as f:
    json.dump({**QUEUE_CONFIG, "BATCH_NAME": BATCH_NAME, "BASE_CONFIG": BASE_CONFIG}, f, indent=2)

LOG_FILE = None  # train_log.txt of the current run, set by run_experiment


def log(*args):
    """Print, and also append to this batch's batch_log.txt and the current run's train_log.txt (the report needs the training log)"""
    msg = " ".join(str(a) for a in args)
    print(msg, flush=True)
    stamp = datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ")
    for path in (os.path.join(BATCH_DIR, "batch_log.txt"), LOG_FILE):
        if path:
            with open(path, "a") as f:
                f.write(stamp + msg + "\n")


ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
log(f"Batch {BATCH_NAME} -> {BATCH_DIR}  (to resume after a disconnect: set BATCH_NAME = \"{BATCH_NAME}\")")
log(f"Device {GPU_NAME} ({VRAM_GB} GB), AMP {AMP_NAME}, {os.cpu_count()} CPUs, {ram_gb:.0f} GB RAM, "
    f"{shutil.disk_usage(DATA_DIR).free / 1e9:.0f} GB free disk")
if AMP_DTYPE == torch.float16:
    log("WARNING: this GPU has no bf16 (e.g. T4); using fp16 + GradScaler, training will be slow.")
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 3. Prepare data from Drive

- Data is read only from `DRIVE_DATA_DIR` on Drive (default `cs604/data/`). Save it there first with `cs604_download_to_drive.ipynb`. If it is not found, you get an error right away.
- Each new session extracts the archives from Drive to the local Colab disk (`DATA_DIR`), and training reads images from there: reading 450,000 small images one by one from Drive would be very slow.
- Images are kept at their original size (longest side 500px) and not resized in advance: the CPU of an A100 runtime decodes fast enough to keep up with the GPU.
- After extraction a marker file is written, so running again in the same Colab session skips this step.

In [ ]:
def prepare_split(split):
    """Extract {split}.tar.gz from Drive to the local Colab disk DATA_DIR/{split}/ and check the image count"""
    list_path = os.path.join(DRIVE_DATA_DIR, f"{split}.txt")
    tar_path = os.path.join(DRIVE_DATA_DIR, f"{split}.tar.gz")
    missing = [p for p in (list_path, tar_path) if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(f"Not found on Drive: {missing}. First run cs604_download_to_drive.ipynb to save the data to Drive"
                                " (run it again after the Phase 2 test set is released)")
    with open(list_path) as f:
        n_expected = sum(1 for line in f if line.strip())
    marker = os.path.join(DATA_DIR, f".{split}_ready")
    if not os.path.exists(marker):
        t0 = time.time()
        log(f"[{split}] extracting {tar_path} ...")
        subprocess.run(["tar", "-xzf", tar_path, "-C", DATA_DIR], check=True)
        n_found = sum(len(files) for _, _, files in os.walk(os.path.join(DATA_DIR, split)))
        assert n_found == n_expected, f"{split}: found {n_found} images, expected {n_expected}"
        open(marker, "w").close()
        log(f"[{split}] {n_found} images ready in {(time.time() - t0) / 60:.1f} min")
    return list_path


def read_list(path):
    items = []
    with open(path) as f:
        for line in f:
            parts = line.split()
            if parts:
                items.append((parts[0], int(parts[1]) if len(parts) > 1 else -1))
    return items


TRAIN_ALL = [] if LOAD_RUN else read_list(prepare_split("train"))  # the training set is not needed when only predicting
PRED_ITEMS = {split: read_list(prepare_split(split)) for split in PREDICT_SPLITS}
log(f"train {len(TRAIN_ALL):,} images | " + " | ".join(f"{s} {len(v):,}" for s, v in PRED_ITEMS.items())
    + f" | {NUM_WORKERS} loader workers")

## 4. Model and fine-tuning methods

- `lora` uses Hugging Face [PEFT](https://github.com/huggingface/peft). After training, the LoRA weights are merged back into the original matrices, so there is no extra cost at inference.
- `adaptformer` follows the [official AdaptFormer code](https://github.com/ShoufaChen/AdaptFormer): bottleneck 64, scale 0.1, parallel to the FFN, up-projection initialized to zero.
- `adapter` is a serial Adapter, placed after the FFN.
- `bitfit` trains only the biases, `partial` trains only the last `PARTIAL_LAST_K` blocks, and `linear` uses only the classifier head.

Trainable params of each method on ViT-B/16 (default config, not including the head; the backbone has 85,798,656 params):

| Method | Trainable params | % of backbone | Extra params at inference |
|---|---|---|---|
| `full` | 85,798,656 | 100% | 0 |
| `lora` (r=8, qkv) | 294,912 | 0.344% | 0 (merged) |
| `adaptformer` / `adapter` (bottleneck 64) | 1,189,632 | 1.387% | 1,189,632 |
| `bitfit` | 102,912 | 0.120% | 0 |
| `partial` (last 2 blocks) | 14,177,280 | 16.5% (more than the 5% the doc suggests) | 0 |
| `linear` | 0 | 0% | 0 |

The 10,000-class classifier head has another 7,690,000 params. The report should say whether it is trained.

In [ ]:
class Adapter(nn.Module):
    """Bottleneck MLP: down → ReLU → up. up starts at zero, so the model output does not change at first after insertion (same as the official AdaptFormer code)"""

    def __init__(self, dim, bottleneck, scale, dropout=0.1):
        super().__init__()
        self.down = nn.Linear(dim, bottleneck)
        self.up = nn.Linear(bottleneck, dim)
        self.drop = nn.Dropout(dropout)
        self.scale = scale
        nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.up(self.drop(F.relu(self.down(x)))) * self.scale


def adaptformer_block_forward(self, x, *args, **kwargs):
    # AdaptFormer: the adapter runs in parallel with the FFN; its input is the residual stream before the FFN branch
    x = x + self.drop_path1(self.ls1(self.attn(self.norm1(x), *args, **kwargs)))
    return x + self.drop_path2(self.ls2(self.mlp(self.norm2(x)))) + self.adapter(x)


def adapter_block_forward(self, x, *args, **kwargs):
    # Serial Adapter: placed after the FFN output, with its own residual
    x = x + self.drop_path1(self.ls1(self.attn(self.norm1(x), *args, **kwargs)))
    h = self.mlp(self.norm2(x))
    return x + self.drop_path2(self.ls2(h + self.adapter(h)))


LORA_MODULES = {"qkv": r"attn\.qkv", "proj": r"attn\.proj", "fc1": r"mlp\.fc1", "fc2": r"mlp\.fc2"}


def is_head(name):
    return re.search(r"(^|\.)head\.(weight|bias)$", name) is not None


def unwrap(net):
    return net.get_base_model() if hasattr(net, "get_base_model") else net


def resolve_config(overrides):
    """Default config + one experiment's overrides → full config for this run"""
    unknown = set(overrides) - set(RUN_KEYS)
    assert not unknown, f"unknown keys in experiment config: {sorted(unknown)}"
    cfg = SimpleNamespace(**{**BASE_CONFIG, **overrides})
    defaults = cfg.METHOD_DEFAULTS[cfg.METHOD]
    cfg.LR = defaults["lr"] if cfg.LR is None else cfg.LR
    cfg.WEIGHT_DECAY = defaults["wd"] if cfg.WEIGHT_DECAY is None else cfg.WEIGHT_DECAY
    cfg.HEAD_LR = cfg.LR if cfg.HEAD_LR is None else cfg.HEAD_LR
    cfg.LAYER_DECAY, cfg.DROP_PATH = defaults["layer_decay"], defaults["drop_path"]
    cfg.ADAPTER_SCALE = defaults.get("adapter_scale", 1.0)
    if cfg.BATCH_SIZE is None:
        cfg.BATCH_SIZE = 256 if (VRAM_GB or 0) >= 40 else 128 if (VRAM_GB or 0) >= 24 else 64
    cfg.ACCUM_STEPS = max(1, cfg.EFFECTIVE_BATCH // cfg.BATCH_SIZE)
    if cfg.TIME_BUDGET_H is None:
        cfg.TIME_BUDGET_H = 96 / VRAM_GB if VRAM_GB else 1.0  # 2 h × 48 GB = 96 GB·h
    return cfg


def create_model(cfg):
    return timm.create_model(cfg.MODEL_NAME, pretrained=True, num_classes=cfg.NUM_CLASSES,
                             drop_path_rate=cfg.DROP_PATH).to(DEVICE)


def apply_method(net, cfg):
    """Freeze params and insert PEFT modules based on METHOD. The returned model may be wrapped by peft"""
    if cfg.METHOD == "full":
        return net.requires_grad_(True)
    net.requires_grad_(False)
    if cfg.METHOD == "lora":
        from peft import LoraConfig, get_peft_model
        pattern = r"blocks\.\d+\.(" + "|".join(LORA_MODULES[t] for t in cfg.LORA_TARGETS) + ")"
        net = get_peft_model(net, LoraConfig(r=cfg.LORA_R, lora_alpha=cfg.LORA_ALPHA, lora_dropout=0.0,
                                             target_modules=pattern, bias="none"))
    elif cfg.METHOD in ("adaptformer", "adapter"):
        forward = adaptformer_block_forward if cfg.METHOD == "adaptformer" else adapter_block_forward
        for blk in net.blocks:
            blk.adapter = Adapter(net.embed_dim, cfg.ADAPTER_DIM, cfg.ADAPTER_SCALE)
            blk.forward = types.MethodType(forward, blk)
    elif cfg.METHOD == "bitfit":
        for n, p in net.named_parameters():
            if n.endswith(".bias"):
                p.requires_grad_(True)
    elif cfg.METHOD == "partial":
        for blk in net.blocks[-cfg.PARTIAL_LAST_K:]:
            blk.requires_grad_(True)
        net.norm.requires_grad_(True)
    unwrap(net).head.requires_grad_(cfg.TRAIN_HEAD and cfg.METHOD != "linear")
    return net


def param_report(net, cfg, ctx, head_stats):
    named = list(net.named_parameters())
    head_trained = any(p.requires_grad for n, p in named if is_head(n))
    n_train = sum(p.numel() for n, p in named if p.requires_grad and not is_head(n))
    source = head_stats.get("head_source", cfg.HEAD_INIT)
    head_mode = ("fine-tuned" if head_trained else "linear probe" if source == "lp"
                 else "class prototypes (not trained)" if source == "proto" else "random, frozen")
    return {"backbone_params": ctx.n_backbone, "head_params": ctx.n_head, "head_mode": head_mode,
            "trainable_excl_head": n_train, "trainable_pct": 100 * n_train / ctx.n_backbone,
            "trainable_incl_head": n_train + (ctx.n_head if head_trained or source == "lp" else 0)}

## 5. Dataset and local holdout set

The official validation set has no labels, so we hold out `HOLDOUT_PER_CLASS` images per class from the training set as a local holdout set, for tuning and ablations. The split is fixed by `SEED`, so it is the same every run.

In [ ]:
def split_holdout(items, k, seed):
    by_class = {}
    for path, label in items:
        by_class.setdefault(label, []).append(path)
    rng = random.Random(seed)
    train, holdout = [], []
    for label in sorted(by_class):
        paths = sorted(by_class[label])
        rng.shuffle(paths)
        holdout += [(p, label) for p in paths[:k]]
        train += [(p, label) for p in paths[k:]]
    return train, holdout


class ImageList(Dataset):
    def __init__(self, items, transform):
        self.items, self.transform = items, transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label = self.items[i]
        with Image.open(os.path.join(DATA_DIR, path)) as im:
            img = im.convert("RGB")
        return self.transform(img), label


BICUBIC = T.InterpolationMode.BICUBIC


def make_ctx(model, cfg):
    """Data setup for this run: train/holdout split, image transforms, normalization values, param counts"""
    data_cfg = timm.data.resolve_model_data_config(model)
    size = data_cfg["input_size"][-1]
    train_items, holdout_items = split_holdout(TRAIN_ALL, cfg.HOLDOUT_PER_CLASS, cfg.SEED) if TRAIN_ALL else ([], [])
    return SimpleNamespace(
        data_cfg=data_cfg, img_size=size, train_items=train_items, holdout_items=holdout_items,
        mean=torch.tensor(data_cfg["mean"], device=DEVICE).view(1, 3, 1, 1) * 255,
        std=torch.tensor(data_cfg["std"], device=DEVICE).view(1, 3, 1, 1) * 255,
        # Only crop/flip on the CPU and output uint8; normalize on the GPU to save CPU work
        train_tf=T.Compose([T.RandomResizedCrop(size, scale=tuple(cfg.RRC_SCALE), interpolation=BICUBIC),
                            T.RandomHorizontalFlip(), T.PILToTensor()]),
        eval_tf=T.Compose([T.Resize(int(size / data_cfg["crop_pct"]), interpolation=BICUBIC),
                           T.CenterCrop(size), T.PILToTensor()]),
        n_backbone=sum(p.numel() for n, p in model.named_parameters() if not is_head(n)),
        n_head=sum(p.numel() for p in model.head.parameters()),
    )


class Loader(DataLoader):
    """Before starting new worker processes, run garbage collection in the main process. Otherwise an old, interrupted DataLoader is copied into the new workers,
    and when it is collected there it prints lots of harmless AssertionError: can only test a child process messages."""

    def __iter__(self):
        if self.num_workers > 0:
            gc.collect()
        return super().__iter__()


def make_loader(items, train, ctx, cfg, shuffle=False):
    return Loader(ImageList(items, ctx.train_tf if train else ctx.eval_tf),
                  batch_size=cfg.BATCH_SIZE if train else 2 * cfg.BATCH_SIZE, shuffle=shuffle, drop_last=train,
                  num_workers=NUM_WORKERS, pin_memory=DEVICE == "cuda",
                  persistent_workers=train and NUM_WORKERS > 0)


def to_input(x, ctx):
    return (x.to(DEVICE, non_blocking=True).float() - ctx.mean) / ctx.std


def autocast():
    return torch.autocast("cuda", dtype=AMP_DTYPE) if DEVICE == "cuda" else contextlib.nullcontext()


def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elif DEVICE == "mps":
        torch.mps.synchronize()


def free_gpu():
    """Free the GPU memory left by the previous run and reset the peak memory stats"""
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

## 6. Classifier head initialization

This is the *Initialization* hint in the doc. If the new 10,000-class head starts random, the early gradients will damage the pretrained features. So we first extract training-set features with the frozen backbone:

1. **Class prototypes**: the mean feature of each class (L2-normalized) is used as the head weight. This is the "Zero-Shot with initialization of classifier head" in the doc (0 trainable params).
2. **Linear probe**: start from the prototypes and train the head on the cached features. This takes only a few seconds. If it is worse than the prototypes on the local holdout set, keep the prototypes.

Within one batch, if the model and data split are the same, features are extracted once and the head is computed once, and later runs reuse them. But **each run's GPU time still includes this time**, so the times in the report are honest. This step is skipped when resuming or only predicting.

In [ ]:
LP_LR, LP_WD, LP_BATCH = 1e-3, 1e-4, 4096
FEATURE_CACHE, HEAD_CACHE = {}, {}


class Clock:
    """Total GPU work time (head init + training + validation during training). Does not include data prep or saving checkpoints"""

    def __init__(self):
        self.used, self._t0 = 0.0, None

    def __enter__(self):
        self._t0 = time.time()
        return self

    def __exit__(self, *exc):
        sync()
        self.used += time.time() - self._t0
        self._t0 = None

    @property
    def elapsed(self):
        return self.used + (time.time() - self._t0 if self._t0 else 0.0)


@torch.inference_mode()
def extract_features(net, items, ctx, cfg, desc):
    net.eval()
    feats, labels = [], []
    for x, y in tqdm(make_loader(items, False, ctx, cfg), desc=desc, leave=False):
        with autocast():
            f = net.forward_head(net.forward_features(to_input(x, ctx)), pre_logits=True)
        feats.append(f.half())
        labels.append(y.to(DEVICE))
    return torch.cat(feats), torch.cat(labels)


@torch.inference_mode()
def class_prototypes(feats, labels, num_classes):
    protos = torch.zeros(num_classes, feats.shape[1], device=feats.device)
    for i in range(0, len(feats), 65536):
        protos.index_add_(0, labels[i:i + 65536], F.normalize(feats[i:i + 65536].float(), dim=1))
    return F.normalize(protos, dim=1)


@torch.inference_mode()
def fit_logit_scale(protos, feats, labels, n=10000):
    """Pick a good temperature for cosine similarity, so the softmax of the prototype head is not too flat or too sharp"""
    idx = torch.randperm(len(feats), device=feats.device)[:n]
    cos = F.normalize(feats[idx].float(), dim=1) @ protos.T
    losses = {s: F.cross_entropy(cos * s, labels[idx]).item() for s in (5, 10, 15, 20, 30, 40, 60, 80)}
    return min(losses, key=losses.get)


@torch.inference_mode()
def topk_accuracy(logits_fn, feats, labels, chunk=4096):
    c1 = c5 = 0
    for i in range(0, len(feats), chunk):
        top5 = logits_fn(feats[i:i + chunk].float()).topk(5, dim=1).indices
        y = labels[i:i + chunk]
        c1 += (top5[:, 0] == y).sum().item()
        c5 += (top5 == y[:, None]).any(dim=1).sum().item()
    return c1 / len(feats), c5 / len(feats)


def linear_probe(feats, labels, weight, bias, cfg):
    head = nn.Linear(feats.shape[1], cfg.NUM_CLASSES).to(DEVICE)
    with torch.no_grad():
        head.weight.copy_(weight)
        head.bias.copy_(bias)
    opt = torch.optim.AdamW(head.parameters(), lr=LP_LR, weight_decay=LP_WD)
    total = cfg.LP_EPOCHS * math.ceil(len(feats) / LP_BATCH)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * s / total)))
    for _ in tqdm(range(cfg.LP_EPOCHS), desc="linear probe", leave=False):
        perm = torch.randperm(len(feats), device=feats.device)
        for i in range(0, len(feats), LP_BATCH):
            idx = perm[i:i + LP_BATCH]
            loss = F.cross_entropy(head(feats[idx].float()), labels[idx], label_smoothing=cfg.LABEL_SMOOTHING)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
    return head.weight.detach(), head.bias.detach()


def cached_features(model, ctx, cfg):
    """Frozen-backbone features for the train / holdout sets. Extracted only once per batch for the same model and split"""
    key = (cfg.MODEL_NAME, cfg.HOLDOUT_PER_CLASS, cfg.SEED)
    if key in FEATURE_CACHE:
        log("Reusing cached backbone features from an earlier experiment in this batch")
        return FEATURE_CACHE[key]
    FEATURE_CACHE.clear()  # keep only one copy to save memory
    t0 = time.time()
    f_tr, y_tr = extract_features(model, ctx.train_items, ctx, cfg, "train features")
    f_ho, y_ho = (extract_features(model, ctx.holdout_items, ctx, cfg, "holdout features")
                  if ctx.holdout_items else (None, None))
    sync()
    FEATURE_CACHE[key] = dict(f_tr=f_tr.cpu(), y_tr=y_tr.cpu(), seconds=time.time() - t0,
                              f_ho=None if f_ho is None else f_ho.cpu(), y_ho=None if y_ho is None else y_ho.cpu())
    log(f"Extracted features for {len(f_tr):,} train / {len(ctx.holdout_items):,} holdout images "
        f"in {FEATURE_CACHE[key]['seconds'] / 60:.1f} min")
    return FEATURE_CACHE[key]


def init_head(model, ctx, cfg):
    """Initialize the classifier head. Returns (head_stats, seconds). The time includes feature extraction, and is counted even when the cache is reused"""
    key = (cfg.MODEL_NAME, cfg.HOLDOUT_PER_CLASS, cfg.SEED, cfg.HEAD_INIT, cfg.LP_EPOCHS, cfg.LABEL_SMOOTHING)
    if key not in HEAD_CACHE:
        feats = cached_features(model, ctx, cfg)
        t0 = time.time()
        f_tr, y_tr = feats["f_tr"].to(DEVICE), feats["y_tr"].to(DEVICE)
        protos = class_prototypes(f_tr, y_tr, cfg.NUM_CLASSES)
        scale = fit_logit_scale(protos, f_tr, y_tr)
        # The linear head works on unnormalized features: divide by the mean feature norm so logits ≈ scale × cosine similarity
        weight = protos * (scale / f_tr.float().norm(dim=1).mean())
        bias = torch.zeros(cfg.NUM_CLASSES, device=DEVICE)
        stats = dict(head_source="proto", proto_logit_scale=scale)
        if feats["f_ho"] is not None:
            f_ho, y_ho = feats["f_ho"].to(DEVICE), feats["y_ho"].to(DEVICE)
            stats["proto_top1"], stats["proto_top5"] = topk_accuracy(lambda f: f @ weight.T, f_ho, y_ho)
            log(f"Prototype head (zero-shot, 0 trainable params): holdout top-1 {stats['proto_top1']:.2%}")
        if cfg.HEAD_INIT == "lp":
            lp_w, lp_b = linear_probe(f_tr, y_tr, weight, bias, cfg)
            if feats["f_ho"] is not None:
                stats["lp_top1"], stats["lp_top5"] = topk_accuracy(lambda f: f @ lp_w.T + lp_b, f_ho, y_ho)
                log(f"Linear-probe head: holdout top-1 {stats['lp_top1']:.2%}")
            if feats["f_ho"] is None or stats["lp_top1"] >= stats["proto_top1"]:
                weight, bias = lp_w, lp_b
                stats["head_source"] = "lp"
            else:
                log("Linear probe is worse than the prototypes on holdout; keeping the prototype head")
        sync()
        HEAD_CACHE[key] = dict(weight=weight.cpu(), bias=bias.cpu(), stats=stats,
                               seconds=feats["seconds"] + time.time() - t0)
    else:
        log("Reusing the cached classifier head from an earlier experiment in this batch")
    cached = HEAD_CACHE[key]
    with torch.no_grad():
        model.head.weight.copy_(cached["weight"].to(DEVICE))
        model.head.bias.copy_(cached["bias"].to(DEVICE))
    log(f"Head initialized from {cached['stats']['head_source']} "
        f"({cached['seconds'] / 60:.1f} min incl. feature extraction, counted in this run's GPU time)")
    return dict(cached["stats"]), cached["seconds"]

## 7. Training

- AdamW, linear warmup for the first `WARMUP_EPOCHS` epochs, then cosine decay. bf16 mixed precision, gradient clipping 1.0, label smoothing 0.1.
- After timing the first 120 steps, if training for `EPOCHS` epochs would go over the time budget, the LR schedule is shrunk to the number of steps that fit.
- At the end of each epoch, the model is evaluated once on the local holdout set and a checkpoint is saved to Drive.

In [ ]:
def build_param_groups(net, cfg):
    """The head uses HEAD_LR. Other params use layer-wise LR decay (on by default only for full). No weight decay on bias, norm and position embeddings"""
    num_layers = len(unwrap(net).blocks) + 1
    groups = {}
    for n, p in net.named_parameters():
        if not p.requires_grad:
            continue
        name = n.replace("base_model.model.", "")
        if is_head(name):
            lr = cfg.HEAD_LR
        else:
            m = re.match(r"blocks\.(\d+)\.", name)
            if m:
                layer = int(m.group(1)) + 1
            elif name.startswith(("cls_token", "pos_embed", "reg_token", "patch_embed")):
                layer = 0
            else:
                layer = num_layers
            lr = cfg.LR * cfg.LAYER_DECAY ** (num_layers - layer)
        wd = 0.0 if p.ndim <= 1 or name in ("cls_token", "pos_embed", "reg_token") else cfg.WEIGHT_DECAY
        groups.setdefault((lr, wd), []).append(p)
    return [{"params": ps, "lr": lr, "weight_decay": wd, "base_lr": lr} for (lr, wd), ps in groups.items()]


@torch.inference_mode()
def evaluate(net, loader, ctx):
    net.eval()
    c1 = c5 = n = 0
    for x, y in loader:
        with autocast():
            top5 = net(to_input(x, ctx)).topk(5, dim=1).indices
        y = y.to(DEVICE)
        c1 += (top5[:, 0] == y).sum().item()
        c5 += (top5 == y[:, None]).any(dim=1).sum().item()
        n += len(y)
    return c1 / n, c5 / n


def save_state(net, path, **extra):
    """Save only the trainable params and the head (LoRA about 30MB, full about 370MB)"""
    state = {n: p.detach().cpu() for n, p in net.named_parameters() if p.requires_grad or is_head(n)}
    torch.save({"state": state, **extra}, path + ".tmp")
    os.replace(path + ".tmp", path)


def train(model, ctx, cfg, run_dir, clock, head_stats, resume_ckpt):
    """Train and return (hist, step, steps_per_epoch, head_stats). The linear method does not train and returns right away"""
    spe = len(ctx.train_items) // (cfg.BATCH_SIZE * cfg.ACCUM_STEPS)
    sched = {"total": 0 if cfg.METHOD == "linear" else cfg.EPOCHS * spe,
             "warmup": max(1, int(cfg.WARMUP_EPOCHS * spe))}
    hist = {"step": [], "loss": [], "lr": [], "epoch": [], "epoch_loss": [], "holdout_top1": [], "steps_per_epoch": spe}
    step, start_epoch = 0, 0
    if not sched["total"]:
        return hist, step, spe, head_stats

    def lr_factor(s):
        if s < sched["warmup"]:
            return (s + 1) / sched["warmup"]
        progress = (s - sched["warmup"]) / max(1, sched["total"] - sched["warmup"])
        return 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(build_param_groups(model, cfg), **({"fused": True} if DEVICE == "cuda" else {}))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)
    if resume_ckpt:
        ck = torch.load(resume_ckpt, map_location="cpu", weights_only=False)
        _, unexpected = model.load_state_dict(ck["state"], strict=False)
        assert not unexpected, f"unexpected keys: {unexpected[:5]}"
        optimizer.load_state_dict(ck["optimizer"])
        scaler.load_state_dict(ck["scaler"])
        step, start_epoch = ck["step"], ck["epoch"] + 1
        sched.update(total=ck["total_steps"], warmup=ck["warmup_steps"])
        clock.used, hist, head_stats = ck["clock"], ck["hist"], ck["head_stats"]
        log(f"Resumed at epoch {start_epoch + 1}, step {step}, {clock.used / 3600:.2f} GPU-hours already used")

    net = torch.compile(model) if cfg.COMPILE and DEVICE == "cuda" else model
    train_loader = make_loader(ctx.train_items, True, ctx, cfg, shuffle=True)
    holdout_loader = make_loader(ctx.holdout_items, False, ctx, cfg) if ctx.holdout_items else None
    budget_s = cfg.TIME_BUDGET_H * 3600
    log(f"Training up to {cfg.EPOCHS} epochs x {spe} steps (batch {cfg.BATCH_SIZE}x{cfg.ACCUM_STEPS}, "
        f"warmup {sched['warmup']} steps)")
    first_step, t_mark, t_start, done = step, 0.0, clock.elapsed, step >= sched["total"]
    for epoch in range(start_epoch, cfg.EPOCHS):
        if done:
            break
        net.train()
        optimizer.zero_grad(set_to_none=True)
        run_loss, run_n, ep_loss, ep_n, t_log = 0.0, 0, 0.0, 0, time.time()
        with clock:
            for i, (x, y) in enumerate(train_loader):
                with autocast():
                    loss = F.cross_entropy(net(to_input(x, ctx)), y.to(DEVICE, non_blocking=True),
                                           label_smoothing=cfg.LABEL_SMOOTHING)
                scaler.scale(loss / cfg.ACCUM_STEPS).backward()
                run_loss += loss.detach()
                run_n += 1
                if (i + 1) % cfg.ACCUM_STEPS:
                    continue
                for g in optimizer.param_groups:
                    g["lr"] = g["base_lr"] * lr_factor(step)
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                step += 1

                # Speed check: shrink the LR schedule so training fits in the time budget
                if step - first_step == 20:
                    t_mark = clock.elapsed
                elif step - first_step == 120:
                    sec_per_step = (clock.elapsed - t_mark) / 100
                    fit = step + int((0.97 * budget_s - clock.elapsed - 30 * (cfg.EPOCHS - epoch)) / sec_per_step)
                    log(f"Speed {cfg.BATCH_SIZE * cfg.ACCUM_STEPS / sec_per_step:.0f} img/s, "
                        f"{sec_per_step * spe / 60:.1f} min/epoch")
                    if fit < sched["total"]:
                        sched["total"] = max(fit, step + 1)
                        sched["warmup"] = min(sched["warmup"], max(1, sched["total"] // 10))
                        log(f"Time budget fits ~{sched['total'] / spe:.1f} epochs: "
                            f"LR schedule compressed to {sched['total']} steps")

                if step % cfg.LOG_EVERY == 0:
                    avg = (run_loss / run_n).item()
                    if not math.isfinite(avg):
                        raise RuntimeError("Loss is NaN/inf; try a lower LR")
                    eta = (sched["total"] - step) * (clock.elapsed - t_start) / (step - first_step)
                    log(f"ep {epoch + 1}/{cfg.EPOCHS} step {step}/{sched['total']} | loss {avg:.4f} | "
                        f"lr {cfg.LR * lr_factor(step):.2e} | {run_n * cfg.BATCH_SIZE / (time.time() - t_log):.0f} img/s | "
                        f"{clock.elapsed / 60:.1f} min | ETA {eta / 60:.0f} min")
                    hist["step"].append(step)
                    hist["loss"].append(avg)
                    hist["lr"].append(cfg.LR * lr_factor(step))
                    ep_loss += avg * run_n
                    ep_n += run_n
                    run_loss, run_n, t_log = 0.0, 0, time.time()
                if step >= sched["total"] or clock.elapsed > 1.05 * budget_s:
                    done = True
                    break
            if run_n:
                ep_loss += float(run_loss)
                ep_n += run_n
            top1 = evaluate(net, holdout_loader, ctx)[0] if holdout_loader else None
        hist["epoch"].append(epoch + 1)
        hist["epoch_loss"].append(ep_loss / max(1, ep_n))
        hist["holdout_top1"].append(top1)
        log(f"== epoch {epoch + 1}: train loss {ep_loss / max(1, ep_n):.4f}"
            + (f", holdout top-1 {top1:.2%}" if top1 is not None else "") + f", {clock.elapsed / 60:.1f} min used")
        if (epoch + 1) % cfg.CKPT_EVERY == 0 or done or epoch + 1 == cfg.EPOCHS:
            save_state(model, os.path.join(run_dir, "last.pt"), optimizer=optimizer.state_dict(),
                       scaler=scaler.state_dict(), step=step, epoch=epoch, total_steps=sched["total"],
                       warmup_steps=sched["warmup"], clock=clock.used, hist=hist, head_stats=head_stats)
    return hist, step, spe, head_stats

## 8. Evaluation, inference latency and submission files

Inference latency is measured with random input and times only the model forward pass (no image decoding), with the same mixed precision as training. In the report, state the GPU model, batch size and precision. With `TTA_FLIP` on, per-image latency counts two forward passes.

In [ ]:
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e1e0d9", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # categorical palette, used in a fixed order


def style_axis(ax, title, xlabel, ylabel):
    ax.set_facecolor(SURFACE)
    ax.set_title(title, loc="left", color=INK, fontsize=11)
    ax.set_xlabel(xlabel, color=MUTED)
    ax.set_ylabel(ylabel, color=MUTED)
    ax.grid(True, color=GRID, linewidth=0.75)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, length=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)


def plot_run_curves(hist, cfg, run_dir):
    if not hist["step"]:
        return
    accs = [(e, 100 * a) for e, a in zip(hist["epoch"], hist["holdout_top1"]) if a is not None]
    fig, axes = plt.subplots(1, 2 if accs else 1, figsize=(11 if accs else 5.5, 3.6), facecolor=SURFACE, squeeze=False)
    ax = axes[0][0]
    ax.plot(hist["step"], hist["loss"], color=SERIES[0], linewidth=1.5, solid_capstyle="round")
    style_axis(ax, f"Training loss ({cfg.METHOD})", "optimizer step", f"cross-entropy (label smoothing {cfg.LABEL_SMOOTHING})")
    if accs:
        ax = axes[0][1]
        ep, acc = zip(*accs)
        ax.plot(ep, acc, color=SERIES[0], linewidth=1.5, marker="o", markersize=6,
                markeredgecolor=SURFACE, markeredgewidth=1.5)
        ax.annotate(f"{acc[-1]:.1f}%", (ep[-1], acc[-1]), textcoords="offset points", xytext=(0, 8),
                    ha="center", color=INK, fontsize=9)
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        style_axis(ax, "Holdout top-1 accuracy", "epoch", "accuracy (%)")
    fig.tight_layout()
    fig.savefig(os.path.join(run_dir, "curves.png"), dpi=150, facecolor=SURFACE)
    plt.close(fig)


@torch.inference_mode()
def measure_latency(net, batch_size, iters, ctx, warmup=10):
    net.eval()
    x = torch.randn(batch_size, 3, ctx.img_size, ctx.img_size, device=DEVICE)
    for _ in range(warmup):
        with autocast():
            net(x)
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        with autocast():
            net(x)
    sync()
    return (time.perf_counter() - t0) / iters


def latency_report(net, ctx, cfg):
    big, k = (256 if DEVICE == "cuda" else 8), (2 if cfg.TTA_FLIP else 1)
    lat_1 = measure_latency(net, 1, 100 if DEVICE == "cuda" else 5, ctx) * k
    lat_b = measure_latency(net, big, 20 if DEVICE == "cuda" else 2, ctx) * k
    setting = f"{GPU_NAME}, {AMP_NAME}, {'with' if cfg.TTA_FLIP else 'no'} flip TTA"
    log(f"Inference latency (batch 1): {lat_1 * 1e3:.2f} ms/img | throughput (batch {big}): {big / lat_b:.0f} img/s | {setting}")
    return dict(latency_ms_bs1=lat_1 * 1e3, throughput_img_s=big / lat_b, latency_setting=setting)


@torch.inference_mode()
def predict(net, items, ctx, cfg):
    net.eval()
    preds = []
    for x, _ in tqdm(make_loader(items, False, ctx, cfg), desc="predict", leave=False):
        x = to_input(x, ctx)
        with autocast():
            logits = net(x).float()
            if cfg.TTA_FLIP:
                logits += net(torch.flip(x, dims=[3])).float()
        preds.append(logits.argmax(dim=1).cpu())
    return torch.cat(preds).tolist()


def make_submission(split, items, net, ctx, cfg, run_name, run_dir):
    """Predict one split and write the CodaBench submission: {split}_team_XX.txt packed into a zip"""
    preds = predict(net, items, ctx, cfg)
    assert len(preds) == len(items) == len({p for p, _ in items}), "every image must appear exactly once"
    assert all(0 <= c < cfg.NUM_CLASSES for c in preds)
    name = f"{split}_team_{TEAM_ID}.txt"
    txt_path = os.path.join(run_dir, name)
    with open(txt_path, "w") as f:
        f.writelines(f"{path} {c}\n" for (path, _), c in zip(items, preds))
    zip_path = os.path.join(run_dir, f"submission_{split}_{run_name}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(txt_path, arcname=name)
    counts = np.bincount(preds, minlength=cfg.NUM_CLASSES)
    log(f"[{split}] {len(preds):,} predictions over {np.count_nonzero(counts):,} classes "
        f"(max {counts.max()} per class) -> {zip_path}")
    return zip_path

## 9. Full pipeline for one experiment

`run_experiment` does these steps in order: build model → init head → insert fine-tuning modules → train → local eval → measure inference latency → write submission files → write `summary.json`. Each run's results are in `cs604/runs/<run name>/` on Drive: `train_log.txt`, `config.json`, `summary.json`, `history.json`, `curves.png`, the submission zip and the weights.

In [ ]:
ARCH_KEYS = ("METHOD", "MODEL_NAME", "NUM_CLASSES", "HEAD_INIT", "TRAIN_HEAD", "LORA_R", "LORA_ALPHA",
             "LORA_TARGETS", "ADAPTER_DIM", "PARTIAL_LAST_K", "METHOD_DEFAULTS")


def run_experiment(overrides, run_name):
    """Run one experiment and return a summary. Finished experiments just load their results; interrupted ones resume from the checkpoint"""
    global LOG_FILE
    cfg = resolve_config(overrides)
    run_dir = os.path.join(OUT_ROOT, "runs", run_name)
    summary_path = os.path.join(run_dir, "summary.json")
    if os.path.exists(summary_path):
        with open(summary_path) as f:
            previous = json.load(f)
        if all(s in previous.get("submissions", {}) for s in PREDICT_SPLITS):
            log(f"== {run_name}: already finished, skipping")
            return previous
    os.makedirs(run_dir, exist_ok=True)
    LOG_FILE = os.path.join(run_dir, "train_log.txt")
    try:
        with open(os.path.join(run_dir, "config.json"), "w") as f:
            json.dump({**vars(cfg), "RUN_NAME": run_name, "BATCH_NAME": BATCH_NAME, "overrides": overrides}, f, indent=2)
        log(f"\n========== {run_name} ==========")
        log(f"{cfg.METHOD}: lr {cfg.LR}, head lr {cfg.HEAD_LR}, wd {cfg.WEIGHT_DECAY}, layer decay {cfg.LAYER_DECAY}, "
            f"epochs {cfg.EPOCHS}, holdout {cfg.HOLDOUT_PER_CLASS}/class, head init {cfg.HEAD_INIT}, "
            f"time budget {cfg.TIME_BUDGET_H:.2f} h")
        random.seed(cfg.SEED)
        np.random.seed(cfg.SEED)
        torch.manual_seed(cfg.SEED)
        free_gpu()

        model = create_model(cfg)
        ctx = make_ctx(model, cfg)
        clock = Clock()
        resume_ckpt = os.path.join(run_dir, "last.pt")
        resume_ckpt = resume_ckpt if os.path.exists(resume_ckpt) else None
        head_stats = {}
        if resume_ckpt:
            log("Found last.pt: resuming this run (head initialization skipped)")
        elif cfg.HEAD_INIT != "random":
            head_stats, seconds = init_head(model, ctx, cfg)
            clock.used += seconds
        model = apply_method(model, cfg).to(DEVICE)
        p0 = param_report(model, cfg, ctx, head_stats)
        log(f"Trainable (excl. head): {p0['trainable_excl_head']:,} = {p0['trainable_pct']:.3f}% of backbone | "
            f"head {ctx.n_head:,} params ({p0['head_mode']})")

        hist, step, spe, head_stats = train(model, ctx, cfg, run_dir, clock, head_stats, resume_ckpt)
        save_state(model, os.path.join(run_dir, "final.pt"), config=vars(cfg), head_stats=head_stats)
        with open(os.path.join(run_dir, "history.json"), "w") as f:
            json.dump(hist, f)
        log(f"Training done: {step} steps, {clock.elapsed / 3600:.2f} GPU-hours (budget {cfg.TIME_BUDGET_H:.2f} h)")

        params = param_report(model, cfg, ctx, head_stats)
        if cfg.METHOD == "lora":
            model = model.merge_and_unload()  # merge LoRA weights into the base weights, so no extra cost at inference
        params["extra_params_at_inference"] = sum(p.numel() for p in model.parameters()) - ctx.n_backbone - ctx.n_head
        results = {}
        if ctx.holdout_items:
            results["holdout_top1"], results["holdout_top5"] = evaluate(
                model, make_loader(ctx.holdout_items, False, ctx, cfg), ctx)
            log(f"Final holdout: top-1 {results['holdout_top1']:.2%}, top-5 {results['holdout_top5']:.2%}")
        plot_run_curves(hist, cfg, run_dir)
        results.update(latency_report(model, ctx, cfg))
        submissions = {split: make_submission(split, items, model, ctx, cfg, run_name, run_dir)
                       for split, items in PRED_ITEMS.items()}

        summary = dict(run=run_name, method=cfg.METHOD, model=cfg.MODEL_NAME, overrides=overrides, **params,
                       **results, **head_stats, gpu=GPU_NAME, vram_gb=VRAM_GB, amp=AMP_NAME,
                       batch=f"{cfg.BATCH_SIZE}x{cfg.ACCUM_STEPS}", epochs_trained=round(step / max(1, spe), 2),
                       steps=step, gpu_hours=clock.elapsed / 3600, budget_hours=cfg.TIME_BUDGET_H,
                       peak_vram_gb=torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else None,
                       submissions=submissions)
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2)
        top1 = f"{summary['holdout_top1']:.2%}" if "holdout_top1" in summary else "–"
        log(f"== {run_name} finished: holdout top-1 {top1} | trainable {params['trainable_excl_head']:,} "
            f"({params['trainable_pct']:.3f}%) | {summary['gpu_hours']:.2f} GPU-h")
        return summary
    finally:
        LOG_FILE = None


def predict_only(run_name):
    """Phase 2: load a trained run and only predict; write the submission files back to that run's folder"""
    global LOG_FILE
    run_dir = os.path.join(OUT_ROOT, "runs", run_name)
    with open(os.path.join(run_dir, "config.json")) as f:
        saved = json.load(f)
    cfg = resolve_config({k: saved[k] for k in ARCH_KEYS if k in saved})
    LOG_FILE = os.path.join(run_dir, "train_log.txt")
    try:
        free_gpu()
        model = create_model(cfg)
        ctx = make_ctx(model, cfg)
        model = apply_method(model, cfg).to(DEVICE)
        ck = torch.load(os.path.join(run_dir, "final.pt"), map_location="cpu", weights_only=False)
        _, unexpected = model.load_state_dict(ck["state"], strict=False)
        assert not unexpected, f"unexpected keys: {unexpected[:5]}"
        log(f"Loaded trained weights from {run_dir}/final.pt")
        if cfg.METHOD == "lora":
            model = model.merge_and_unload()
        summary_path = os.path.join(run_dir, "summary.json")
        summary = {"run": run_name, "method": cfg.METHOD}
        if os.path.exists(summary_path):
            with open(summary_path) as f:
                summary = json.load(f)
        summary.update(latency_report(model, ctx, cfg))
        summary.setdefault("submissions", {}).update(
            {split: make_submission(split, items, model, ctx, cfg, run_name, run_dir) for split, items in PRED_ITEMS.items()})
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2)
        return summary
    finally:
        LOG_FILE = None

## 10. Run the experiment queue

Runs in the order of `EXPERIMENTS`. If the local holdout accuracy of the zero-shot baseline is below 25% (the doc reports about 43.7%), the data pipeline is likely broken, so the queue stops to avoid wasting hours of GPU time.

In [ ]:
RESULTS = []
if LOAD_RUN:
    RESULTS.append(predict_only(LOAD_RUN))
else:
    for i, exp in enumerate(EXPERIMENTS, 1):
        exp = dict(exp)
        name = exp.pop("RUN_NAME", None) or f"{BATCH_NAME}_{i}_{exp.get('METHOD', METHOD)}"
        result = run_experiment(exp, name)
        RESULTS.append(result)
        if result["method"] == "linear" and result.get("proto_top1") is not None and result["proto_top1"] < 0.25:
            raise RuntimeError(f"Zero-shot holdout top-1 is only {result['proto_top1']:.1%} (expected ~43.7%): "
                               "the data pipeline is probably broken, stopping the queue. Please check the log.")

## 11. Automatic final training (optional, off by default)

Runs only when `RUN_FINAL = True`: among the trained runs (excluding `linear`), it picks the method with the best local holdout top-1 (or the one set in `FINAL_METHOD`), keeps its config, and applies `FINAL_OVERRIDES` (by default 15 epochs, all 45 images per class). It does not tune the LR, epochs, etc. based on the short runs, so it is better to check the short-run results first, put your final config in `EXPERIMENTS`, and run a new batch.

In [ ]:
if RUN_FINAL and not LOAD_RUN:
    trained = [r for r in RESULTS if r.get("steps") and r.get("holdout_top1") is not None]
    ranking = ", ".join(f"{r['method']} {r['holdout_top1']:.2%}" for r in sorted(trained, key=lambda r: -r["holdout_top1"]))
    if FINAL_METHOD:
        pick = next((r for r in trained if r["method"] == FINAL_METHOD), None)
        base = dict(pick["overrides"]) if pick else {"METHOD": FINAL_METHOD}
    else:
        base = dict(max(trained, key=lambda r: r["holdout_top1"])["overrides"]) if trained else None
    if base is None:
        log("RUN_FINAL: no trained experiment with holdout accuracy to choose from; skipping the final run")
    else:
        final_overrides = {**base, **FINAL_OVERRIDES}
        log(f"\nFinal run with {final_overrides}  (holdout ranking: {ranking or 'n/a'})")
        RESULTS.append(run_experiment(final_overrides, f"{BATCH_NAME}_final_{final_overrides.get('METHOD', METHOD)}"))

## 12. Compare results (for the report)

In [ ]:
def fmt_pct(v):
    return f"{v:.2%}" if isinstance(v, (int, float)) else "–"


header = ("| Run | Method | Holdout top-1 | Holdout top-5 | Trainable params (excl. head) | Classifier head | "
          "Latency bs1 (ms) | GPU-hours | Epochs |\n|---|---|---|---|---|---|---|---|---|")
rows = [f"| {r['run']} | {r['method']} | {fmt_pct(r.get('holdout_top1'))} | {fmt_pct(r.get('holdout_top5'))} | "
        f"{r.get('trainable_excl_head', 0):,} ({r.get('trainable_pct', 0):.3f}%) | {r.get('head_mode', '–')} | "
        f"{r.get('latency_ms_bs1', float('nan')):.2f} | {r.get('gpu_hours', 0):.2f} | {r.get('epochs_trained', 0)} |"
        for r in RESULTS]
table = header + "\n" + "\n".join(rows)
with open(os.path.join(BATCH_DIR, "comparison.md"), "w") as f:
    f.write(table + "\n")
with open(os.path.join(BATCH_DIR, "results.json"), "w") as f:
    json.dump(RESULTS, f, indent=2)
log("\n" + table)
try:
    from IPython.display import Markdown, display
    display(Markdown(table))
except ImportError:
    pass

# Comparison plot: one line per trained experiment (max 4), x-axis is always epoch
curves = []
for r in RESULTS:
    path = os.path.join(OUT_ROOT, "runs", r["run"], "history.json")
    if r.get("steps") and os.path.exists(path):
        with open(path) as f:
            curves.append((r["run"].replace(f"{BATCH_NAME}_", ""), json.load(f)))
if len(curves) > len(SERIES):
    log(f"Comparison plot shows the first {len(SERIES)} trained runs only")
if curves:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor=SURFACE)
    ends = []
    for (label, h), color in zip(curves, SERIES):
        spe = h.get("steps_per_epoch") or 1
        axes[0].plot([s / spe for s in h["step"]], h["loss"], color=color, linewidth=1.5, label=label)
        acc = [(e, 100 * a) for e, a in zip(h["epoch"], h["holdout_top1"]) if a is not None]
        if acc:
            e, a = zip(*acc)
            axes[1].plot(e, a, color=color, linewidth=1.5, marker="o", markersize=6,
                         markeredgecolor=SURFACE, markeredgewidth=1.5, label=label)
            ends.append((a[-1], e[-1]))
    # value labels at line ends: sort by value and shift up/down so they don't overlap
    for rank, (value, epoch) in enumerate(sorted(ends, reverse=True)):
        axes[1].annotate(f"{value:.1f}%", (epoch, value), textcoords="offset points",
                         xytext=(6, 10 * (len(ends) - 1) / 2 - 10 * rank), va="center", color=INK, fontsize=9)
    style_axis(axes[0], "Training loss", "epoch", "cross-entropy")
    style_axis(axes[1], "Holdout top-1 accuracy", "epoch", "accuracy (%)")
    axes[1].xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    for ax in axes:
        if ax.get_legend_handles_labels()[0]:
            ax.legend(frameon=False, labelcolor=INK, fontsize=9)
    fig.tight_layout()
    fig.savefig(os.path.join(BATCH_DIR, "comparison.png"), dpi=150, facecolor=SURFACE)
    plt.show()

## 13. Auto disconnect (optional)

All results are already saved to Drive at this point. `DISCONNECT_WHEN_DONE = True` releases the runtime after making sure the data is written to Drive, to stop using compute units. Don't turn it on if you still want to run more experiments in the same session.

In [ ]:
if IN_COLAB and DISCONNECT_WHEN_DONE:
    from google.colab import drive, runtime
    log("All done; releasing the runtime ...")
    drive.flush_and_unmount()  # make sure all results are uploaded to Drive
    runtime.unassign()

## Next steps

- **Submitting**: each experiment's `submission_val_*.zip` is in `cs604/runs/<run name>/` on Drive. Download it, upload it to CodaBench (max 5 per day), and write down the score.
- **Full training**: after picking the final settings from the short runs, put them in `EXPERIMENTS` and run a new batch, e.g. `EXPERIMENTS = [dict(METHOD="full", EPOCHS=15, HOLDOUT_PER_CLASS=0)]` (it is shortened automatically if it goes over the time budget; `HOLDOUT_PER_CLASS=0` means all 45 images per class are used for training, so check the accuracy on CodaBench).
- **Writing the report**: `comparison.md` and `comparison.png` in `cs604/batches/<BATCH_NAME>/` can go straight into the report. The `train_log.txt` in each run folder is the training log to attach.
- **Ablations**: change `EXPERIMENTS` to queue a new set of experiments, e.g. head init (`HEAD_INIT` = random / proto / lp), LoRA rank and where it is inserted (`LORA_R`, `LORA_TARGETS`), adapter size, learning rate, etc. Example: `dict(METHOD="lora", EPOCHS=3, WARMUP_EPOCHS=0.3, LORA_R=16, LORA_TARGETS=["qkv", "proj", "fc1", "fc2"])`.
- **Phase 2 (test set released on 10/31)**: first run `cs604_download_to_drive.ipynb` again to save the test set to Drive. Then set `LOAD_RUN = "<final training run name>"` and `PREDICT_SPLITS = ["test"]`, and "Run all" to create `test_team_XX.txt`. No retraining is needed, and the training set is not downloaded.